# 00_bootstrap_universe

Builds the point-in-time S&P 500 universe table and trading calendar
for the Precursor financial ML pipeline.

**Run this notebook ONCE manually before any other bootstrap script.**
All downstream notebooks depend on `precursor.bronze.universe`.

In [0]:
%pip install \
    "numpy<2.0" \
    lxml html5lib beautifulsoup4 \
    pyluach korean_lunar_calendar toolz tzdata \
    exchange-calendars pandas-market-calendars \
    --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import re
import logging
import requests
import pandas as pd
import numpy as np
import pandas_market_calendars as mcal

from datetime import datetime, date, timedelta
from typing import Optional

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, BooleanType,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.universe")

START_DATE = "2020-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

logger.info("START_DATE=%s  END_DATE=%s", START_DATE, END_DATE)

INFO:precursor.universe:START_DATE=2020-01-01  END_DATE=2026-04-30


## Cell 3 — Create catalog and schemas

In [0]:
def _create_catalog_and_schemas() -> None:
    """Create the precursor catalog and bronze/silver/gold schemas if they don't exist."""
    spark.sql("CREATE CATALOG IF NOT EXISTS precursor")
    logger.info("Catalog 'precursor' ready.")

    for schema in ("bronze", "silver", "gold"):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS precursor.{schema}")
        logger.info("Schema 'precursor.%s' ready.", schema)

_create_catalog_and_schemas()

INFO:precursor.universe:Catalog 'precursor' ready.
INFO:precursor.universe:Schema 'precursor.bronze' ready.
INFO:precursor.universe:Schema 'precursor.silver' ready.
INFO:precursor.universe:Schema 'precursor.gold' ready.


## Cell 4 — Fetch current S&P 500 members

In [0]:
_WIKIPEDIA_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Dot-to-slash replacements required by Alpaca's ticker format
_TICKER_REPLACEMENTS = {"BRK.B": "BRK/B", "BF.B": "BF/B"}

_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def _fetch_wikipedia_html() -> str:
    """Fetch the S&P 500 Wikipedia page HTML with a browser user-agent.

    Returns:
        Raw HTML string.

    Raises:
        RuntimeError: If the HTTP request fails.
    """
    response = requests.get(_WIKIPEDIA_URL, headers=_HEADERS, timeout=30)
    if not response.ok:
        raise RuntimeError(f"Wikipedia request failed: HTTP {response.status_code}")
    return response.text


def fetch_current_sp500() -> pd.DataFrame:
    """Fetch the current S&P 500 constituent list from Wikipedia.

    Parses Table 0 of the S&P 500 Wikipedia page, which contains
    the current index members.

    Returns:
        DataFrame with columns:
            ticker (str), company_name (str), sector (str), date_added (str)
    """
    try:
        html = _fetch_wikipedia_html()
        tables = pd.read_html(html, attrs={"id": "constituents"})
        df = tables[0]
    except Exception as exc:
        raise RuntimeError(f"Failed to fetch current S&P 500 members: {exc}") from exc

    df = df.rename(columns={
        "Symbol":      "ticker",
        "Security":    "company_name",
        "GICS Sector": "sector",
        "Date added":  "date_added",
    })

    df["ticker"] = (
        df["ticker"]
        .str.strip()
        .replace(_TICKER_REPLACEMENTS)
    )
    df["company_name"] = df["company_name"].str.strip()
    df["sector"]       = df["sector"].str.strip()
    df["date_added"]   = df["date_added"].astype(str).str.strip()

    df = df[["ticker", "company_name", "sector", "date_added"]]
    logger.info("Current S&P 500 members fetched: %d tickers.", len(df))
    return df

INFO:py4j.clientserver:Received command c on object id p0


## Cell 5 — Fetch historical S&P 500 changes

In [0]:
def fetch_sp500_changes() -> pd.DataFrame:
    """Fetch the historical S&P 500 addition/removal log from Wikipedia.

    Parses Table 1 of the S&P 500 Wikipedia page.  Column names on
    Wikipedia vary over time, so columns are addressed by position.

    Returns:
        DataFrame with columns:
            date (datetime), added_ticker (str|None), added_name (str|None),
            removed_ticker (str|None), removed_name (str|None), reason (str|None)
        Only rows with dates >= 2019-01-01 are returned.
    """
    try:
        html = _fetch_wikipedia_html()
        tables = pd.read_html(html, attrs={"id": "changes"})
        raw = tables[0]
    except Exception:
        # Fallback: grab the second table by index if id lookup fails
        try:
            html = _fetch_wikipedia_html()
            tables = pd.read_html(html)
            raw = tables[1]
        except Exception as exc:
            raise RuntimeError(f"Failed to fetch S&P 500 change history: {exc}") from exc

    # Flatten multi-level column headers that Wikipedia sometimes emits
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [" ".join(str(c) for c in col).strip() for col in raw.columns]

    cols = raw.columns.tolist()

    # Map positional columns to canonical names (at least 5 columns expected)
    rename = {}
    if len(cols) >= 1:
        rename[cols[0]] = "date"
    if len(cols) >= 2:
        rename[cols[1]] = "added_ticker"
    if len(cols) >= 3:
        rename[cols[2]] = "added_name"
    if len(cols) >= 4:
        rename[cols[3]] = "removed_ticker"
    if len(cols) >= 5:
        rename[cols[4]] = "removed_name"
    if len(cols) >= 6:
        rename[cols[5]] = "reason"

    df = raw.rename(columns=rename)

    # -- Clean date column --------------------------------------------------
    df["date"] = (
        df["date"]
        .astype(str)
        .str.replace(r"\[.*", "", regex=True)   # strip footnote markers
        .str.strip()
    )
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    # -- Clean ticker columns -----------------------------------------------
    def _clean_ticker(series: pd.Series) -> pd.Series:
        cleaned = (
            series.astype(str)
            .str.strip()
            .str.replace(".", "/", regex=False)
        )
        return cleaned.where(cleaned.str.len() > 0, other=None)

    for col in ("added_ticker", "removed_ticker"):
        if col in df.columns:
            df[col] = _clean_ticker(df[col])
            # "nan" strings → None
            df[col] = df[col].where(df[col] != "nan", other=None)

    for col in ("added_name", "removed_name"):
        if col not in df.columns:
            df[col] = None

    if "reason" not in df.columns:
        df["reason"] = None

    # -- Filter to history we actually need --------------------------------
    cutoff = pd.Timestamp("2019-01-01")
    df = df[df["date"] >= cutoff].copy()

    logger.info("S&P 500 historical changes loaded: %d rows after 2019-01-01.", len(df))
    return df[["date", "added_ticker", "added_name",
               "removed_ticker", "removed_name", "reason"]]

## Cell 6 — Build trading calendar

In [0]:
def build_trading_calendar(start_date: str, end_date: str) -> pd.DataFrame:
    """Build a daily calendar that flags NYSE trading days.

    Args:
        start_date: ISO date string "YYYY-MM-DD".
        end_date:   ISO date string "YYYY-MM-DD".

    Returns:
        DataFrame with columns:
            date (date), is_trading_day (bool)
        One row per calendar day in [start_date, end_date].
    """
    nyse     = mcal.get_calendar("NYSE")
    schedule = nyse.schedule(start_date=start_date, end_date=end_date)
    trading_days = set(schedule.index.normalize().date)

    all_days = pd.date_range(start=start_date, end=end_date, freq="D")
    calendar_df = pd.DataFrame({"date": all_days.date})
    calendar_df["is_trading_day"] = calendar_df["date"].isin(trading_days)

    total_calendar = len(calendar_df)
    total_trading  = calendar_df["is_trading_day"].sum()
    logger.info(
        "Trading calendar built: %d calendar days, %d trading days.",
        total_calendar, total_trading,
    )
    return calendar_df

## Cell 7 — Reconstruct point-in-time membership

In [0]:
def reconstruct_universe(
    current_members: pd.DataFrame,
    changes: pd.DataFrame,
    calendar: pd.DataFrame,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """Reconstruct a point-in-time S&P 500 membership table.

    Starting from today's index and working backwards through the change
    log, assigns in_index=True/False for every ticker on every calendar day.

    Args:
        current_members: Output of fetch_current_sp500().
        changes:         Output of fetch_sp500_changes().
        calendar:        Output of build_trading_calendar().
        start_date:      ISO date string "YYYY-MM-DD".
        end_date:        ISO date string "YYYY-MM-DD".

    Returns:
        DataFrame with columns:
            date (date), ticker (str), company_name (str), sector (str),
            in_index (bool), is_trading_day (bool)
        One row per ticker per calendar day in [start_date, end_date].
    """
    start_dt = pd.Timestamp(start_date).date()
    end_dt   = pd.Timestamp(end_date).date()

    # -- Build metadata lookup (name / sector) ---------------------------------
    meta: dict[str, dict] = {}
    for _, row in current_members.iterrows():
        meta[row["ticker"]] = {
            "company_name": row["company_name"],
            "sector":       row["sector"],
        }

    # Fill metadata for historical tickers from change log
    for _, row in changes.iterrows():
        for ticker, name_col in [
            (row.get("added_ticker"),   row.get("added_name")),
            (row.get("removed_ticker"), row.get("removed_name")),
        ]:
            if ticker and pd.notna(ticker) and ticker not in meta:
                meta[ticker] = {
                    "company_name": name_col if (name_col and pd.notna(name_col)) else "",
                    "sector":       "",
                }

    current_set = set(current_members["ticker"])

    # Collect every ticker ever referenced
    all_tickers: set[str] = set(current_set)
    for col in ("added_ticker", "removed_ticker"):
        valid = changes[col].dropna()
        valid = valid[valid.str.len() > 0]
        all_tickers.update(valid.tolist())

    # -- Reconstruct per-ticker membership intervals --------------------------
    # For each ticker we compute the sorted list of (date, in_index) breakpoints
    # and then decide, for each calendar day, whether the ticker was in the index.

    calendar_dates = calendar["date"].tolist()  # list[date]
    is_trading     = dict(zip(calendar["date"], calendar["is_trading_day"]))

    rows: list[dict] = []
    tickers_entered: set[str] = set()
    tickers_left:    set[str] = set()

    for ticker in all_tickers:
        # Gather all change events for this ticker
        events: list[tuple[date, bool]] = []   # (effective_date, in_index_from_that_day)

        ticker_adds = changes[changes["added_ticker"] == ticker]
        ticker_rems = changes[changes["removed_ticker"] == ticker]

        for _, row in ticker_adds.iterrows():
            event_date = row["date"].date()
            events.append((event_date, True))   # in_index from this date onward

        for _, row in ticker_rems.iterrows():
            event_date = row["date"].date()
            events.append((event_date, False))  # out of index from this date onward

        # Sort chronologically (oldest first) so we can replay forward
        events.sort(key=lambda x: x[0])

        # Default state: what we observe TODAY
        default_in_index = ticker in current_set

        # Determine status on each calendar day using the sorted event list.
        # We step through events to find the applicable state at each date.
        # A "removed" event on date X means the ticker was removed starting X.
        # An "added" event on date X means the ticker was added starting X.
        # We reconstruct backwards: starting from default_in_index, each event
        # is a breakpoint.  Before the earliest event in our window, we infer
        # the pre-breakpoint state by inverting the first event.

        def status_on(query_date: date) -> bool:
            """Return in_index for query_date given sorted events and default."""
            # Find the last event on or before query_date
            applicable = [e for e in events if e[0] <= query_date]
            if applicable:
                return applicable[-1][1]
            # No event on or before query_date — infer pre-history state
            if events:
                # State just before the first event is the opposite of that event
                first_state = events[0][1]
                return not first_state
            # No events at all: use current membership as the universal default
            return default_in_index

        # Determine whether this ticker appears anywhere in our window
        state_at_start = status_on(start_dt)
        state_at_end   = status_on(end_dt)
        ever_in_window = state_at_start or state_at_end or any(
            start_dt <= e[0] <= end_dt for e in events
        )

        if not ever_in_window:
            continue  # ticker was never in the index during our window

        # Track joiners / leavers
        had_addition_in_window = any(
            start_dt <= e[0] <= end_dt and e[1] is True for e in events
        )
        had_removal_in_window = any(
            start_dt <= e[0] <= end_dt and e[1] is False for e in events
        )
        if had_addition_in_window:
            tickers_entered.add(ticker)
        if had_removal_in_window:
            tickers_left.add(ticker)

        ticker_meta = meta.get(ticker, {"company_name": "", "sector": ""})

        for d in calendar_dates:
            rows.append({
                "date":          d,
                "ticker":        ticker,
                "company_name":  ticker_meta["company_name"],
                "sector":        ticker_meta["sector"],
                "in_index":      status_on(d),
                "is_trading_day": is_trading[d],
            })

    universe_df = pd.DataFrame(rows)

    logger.info("Total unique tickers in universe: %d", len(all_tickers))
    logger.info("Tickers that ENTERED index since %s: %d", start_date, len(tickers_entered))
    logger.info("Tickers that LEFT index since %s: %d",   start_date, len(tickers_left))
    logger.info("Total rows generated: %d", len(universe_df))
    return universe_df

## Cell 8 — Validate universe

In [0]:
def validate_universe(universe_df: pd.DataFrame) -> bool:
    """Run data-quality checks on the reconstructed universe DataFrame.

    Args:
        universe_df: Output of reconstruct_universe().

    Returns:
        True if all checks pass.

    Raises:
        ValueError: If any check fails, with a description of the failure.
    """
    failures: list[str] = []

    def _check(name: str, passed: bool, detail: str = "") -> None:
        status = "PASS" if passed else "FAIL"
        msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
        logger.info(msg)
        if not passed:
            failures.append(f"{name}: {detail}")

    # 1. No null tickers
    null_tickers = universe_df["ticker"].isna().sum()
    _check("No null tickers", null_tickers == 0, f"{null_tickers} nulls found")

    # 2. No null dates
    null_dates = universe_df["date"].isna().sum()
    _check("No null dates", null_dates == 0, f"{null_dates} nulls found")

    # 3. Date range covers full window
    min_date = pd.Timestamp(universe_df["date"].min())
    max_date = pd.Timestamp(universe_df["date"].max())
    _check(
        "Date range starts on or before 2020-01-01",
        min_date <= pd.Timestamp("2020-01-01"),
        f"min date = {min_date.date()}",
    )
    _check(
        "Date range ends on or after today",
        max_date >= pd.Timestamp(END_DATE),
        f"max date = {max_date.date()}, expected >= {END_DATE}",
    )

    # 4. At least 400 members in index on any trading day
    trading_days = universe_df[universe_df["is_trading_day"]]
    daily_counts = trading_days[trading_days["in_index"]].groupby("date")["ticker"].count()
    min_members  = int(daily_counts.min()) if len(daily_counts) > 0 else 0
    _check(
        "At least 400 members per trading day",
        min_members >= 400,
        f"minimum observed = {min_members}",
    )

    # 5. No duplicate (date, ticker) pairs
    dup_count = universe_df.duplicated(subset=["date", "ticker"]).sum()
    _check("No duplicate (date, ticker) pairs", dup_count == 0, f"{dup_count} duplicates found")

    # 6. Total unique tickers in reasonable range
    unique_tickers = universe_df["ticker"].nunique()
    _check(
        "Unique tickers between 500 and 1500",
        500 <= unique_tickers <= 1500,
        f"found {unique_tickers}",
    )

    if failures:
        raise ValueError("Universe validation FAILED:\n" + "\n".join(failures))

    logger.info("All validation checks PASSED.")
    return True

## Cell 9 — Write to Delta table

In [0]:
def write_universe_to_delta(universe_df: pd.DataFrame) -> None:
    """Write the universe DataFrame to precursor.bronze.universe as a Delta table.

    Args:
        universe_df: Validated output of reconstruct_universe().
    """
    schema = StructType([
        StructField("date",          DateType(),    nullable=False),
        StructField("ticker",        StringType(),  nullable=False),
        StructField("company_name",  StringType(),  nullable=True),
        StructField("sector",        StringType(),  nullable=True),
        StructField("in_index",      BooleanType(), nullable=False),
        StructField("is_trading_day", BooleanType(), nullable=False),
    ])

    # Convert python date objects to string so createDataFrame handles them cleanly
    write_df = universe_df.copy()
    write_df["date"] = pd.to_datetime(write_df["date"]) 

    spark_df: DataFrame = spark.createDataFrame(write_df, schema=schema)

    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.universe")
    )

    row_count = spark.table("precursor.bronze.universe").count()
    logger.info("precursor.bronze.universe written: %d rows.", row_count)

    display(spark.sql("""
        SELECT *
        FROM precursor.bronze.universe
        WHERE is_trading_day = true
          AND in_index = true
        ORDER BY date DESC
        LIMIT 20
    """))

## Cell 11 — Main execution

In [0]:
_start_time = datetime.now()
logger.info("=== precursor.00_bootstrap_universe START at %s ===", _start_time.isoformat())

try:
    # Step 1: Fetch current members
    logger.info("Step 1/6 — Fetching current S&P 500 members …")
    current_members = fetch_current_sp500()

    # Step 2: Fetch historical changes
    logger.info("Step 2/6 — Fetching historical S&P 500 changes …")
    changes = fetch_sp500_changes()

    # Step 3: Build trading calendar
    logger.info("Step 3/6 — Building NYSE trading calendar …")
    calendar_df = build_trading_calendar(START_DATE, END_DATE)

    # Step 4: Reconstruct point-in-time universe
    logger.info("Step 4/6 — Reconstructing point-in-time universe …")
    universe_df = reconstruct_universe(
        current_members=current_members,
        changes=changes,
        calendar=calendar_df,
        start_date=START_DATE,
        end_date=END_DATE,
    )

    # Step 5: Validate
    logger.info("Step 5/6 — Validating universe …")
    validate_universe(universe_df)

    # Step 6: Write to Delta
    logger.info("Step 6/6 — Writing to precursor.bronze.universe …")
    write_universe_to_delta(universe_df)

    # Summary
    print_summary(universe_df, calendar_df)

except Exception as exc:
    logger.error("Bootstrap failed: %s", exc, exc_info=True)
    raise

_end_time = datetime.now()
_elapsed  = (_end_time - _start_time).total_seconds() / 60
logger.info(
    "=== precursor.00_bootstrap_universe END at %s (%.2f min) ===",
    _end_time.isoformat(), _elapsed,
)

INFO:precursor.universe:=== precursor.00_bootstrap_universe START at 2026-04-30T21:58:22.085499 ===
INFO:precursor.universe:Step 1/6 — Fetching current S&P 500 members …
INFO:precursor.universe:Current S&P 500 members fetched: 503 tickers.
INFO:precursor.universe:Step 2/6 — Fetching historical S&P 500 changes …
INFO:precursor.universe:S&P 500 historical changes loaded: 148 rows after 2019-01-01.
INFO:precursor.universe:Step 3/6 — Building NYSE trading calendar …
INFO:precursor.universe:Trading calendar built: 2312 calendar days, 1590 trading days.
INFO:precursor.universe:Step 4/6 — Reconstructing point-in-time universe …
INFO:precursor.universe:Total unique tickers in universe: 636
INFO:precursor.universe:Tickers that ENTERED index since 2020-01-01: 109
INFO:precursor.universe:Tickers that LEFT index since 2020-01-01: 111
INFO:precursor.universe:Total rows generated: 1417256
INFO:precursor.universe:Step 5/6 — Validating universe …
INFO:precursor.universe:[PASS] No null tickers — 0 null

date,ticker,company_name,sector,in_index,is_trading_day
2026-04-30,KR,Kroger,Consumer Staples,true,true
2026-04-30,VRSK,Verisk Analytics,Industrials,true,true
2026-04-30,EFX,Equifax,Industrials,true,true
2026-04-30,DLR,Digital Realty,Real Estate,true,true
2026-04-30,HSY,Hershey Company (The),Consumer Staples,true,true
2026-04-30,UPS,United Parcel Service,Industrials,true,true
2026-04-30,GOOGL,Alphabet Inc. (Class A),Communication Services,true,true
2026-04-30,VTR,Ventas,Real Estate,true,true
2026-04-30,UBER,Uber,Industrials,true,true
2026-04-30,DPZ,Domino's,Consumer Discretionary,true,true


ERROR:precursor.universe:Bootstrap failed: name 'print_summary' is not defined
Traceback (most recent call last):
  File "/home/spark-745024ac-a7f1-4ae0-bf6e-2b/.ipykernel/3712/command-8491682681197065-2334372601", line 36, in <module>
    print_summary(universe_df, calendar_df)
    ^^^^^^^^^^^^^
NameError: name 'print_summary' is not defined


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8491682681197065>, line 36
     33     write_universe_to_delta(universe_df)
     35     # Summary
---> 36     print_summary(universe_df, calendar_df)
     38 except Exception as exc:
     39     logger.error("Bootstrap failed: %s", exc, exc_info=True)

NameError: name 'print_summary' is not defined

## Cell 10 — Final summary

In [0]:
def print_summary(
    universe_df: pd.DataFrame,
    calendar_df: pd.DataFrame,
) -> None:
    """Print a human-readable summary of the universe build to notebook output.

    Args:
        universe_df: Output of reconstruct_universe().
        calendar_df: Output of build_trading_calendar().
    """
    latest_date   = universe_df["date"].max()
    current_count = universe_df[
        (universe_df["date"] == latest_date) & universe_df["in_index"]
    ]["ticker"].nunique()

    unique_tickers = universe_df["ticker"].nunique()
    total_rows     = len(universe_df)
    total_trading  = int(calendar_df["is_trading_day"].sum())

    # Tickers that joined and left during the window
    added_tickers = set()
    left_tickers  = set()
    for ticker, grp in universe_df.groupby("ticker"):
        grp_sorted = grp.sort_values("date")
        states     = grp_sorted["in_index"].tolist()
        # entered: first row is False but later becomes True
        if not states[0] and any(states):
            added_tickers.add(ticker)
        # left: last row is False and earlier rows had True
        if not states[-1] and any(states):
            left_tickers.add(ticker)

    # Sample 5 tickers with date range
    sample_tickers = universe_df[universe_df["in_index"]]["ticker"].drop_duplicates().head(5).tolist()
    sample_lines   = []
    for t in sample_tickers:
        t_df = universe_df[universe_df["ticker"] == t]
        sample_lines.append(
            f"  {t:<10} {t_df['date'].min()}  →  {t_df['date'].max()}"
        )

    #Latest members
    latest_members = universe_df[(universe_df["date"] == latest_date) & (universe_df["in_index"])]

    # Sector breakdown (current members only)
    sector_counts = (
        latest_members[latest_members["sector"].str.strip() != ""]
        .groupby("sector")["ticker"]
        .nunique()
        .sort_values(ascending=False)
    )
    sector_lines = [f"  {sector:<45} {count}" for sector, count in sector_counts.items()]

    separator = "=" * 60
    print(separator)
    print("  PRECURSOR — UNIVERSE BUILD SUMMARY")
    print(separator)
    print(f"  Total unique tickers ever in universe : {unique_tickers}")
    print(f"  Current members (as of {latest_date})  : {current_count}")
    print(f"  Tickers that joined since 2020        : {len(added_tickers)}")
    print(f"  Tickers that left since 2020          : {len(left_tickers)}")
    print(f"  Total NYSE trading days               : {total_trading}")
    print(f"  Total rows in universe table          : {total_rows:,}")
    print()
    print("  Sample tickers — full date range:")
    print("\n".join(sample_lines))
    print()
    print("  Sectors (current members):")
    print("\n".join(sector_lines))
    print(separator)

In [0]:
print_summary(universe_df, calendar_df) 

  PRECURSOR — UNIVERSE BUILD SUMMARY
  Total unique tickers ever in universe : 613
  Current members (as of 2026-04-30)  : 502
  Tickers that joined since 2020        : 109
  Tickers that left since 2020          : 111
  Total NYSE trading days               : 1590
  Total rows in universe table          : 1,417,256

  Sample tickers — full date range:
  CVNA       2020-01-01  →  2026-04-30
  CPRI       2020-01-01  →  2026-04-30
  TFX        2020-01-01  →  2026-04-30
  UDR        2020-01-01  →  2026-04-30
  FLIR       2020-01-01  →  2026-04-30

  Sectors (current members):
  Industrials                                   79
  Financials                                    76
  Information Technology                        73
  Health Care                                   58
  Consumer Discretionary                        48
  Consumer Staples                              36
  Real Estate                                   31
  Utilities                                     31
  Materials 